In [119]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd 


In [120]:
# nous allons vérifier que nous accédons sans soucis à nos identifiants .env
load_dotenv()

CLIENT_ID = os.getenv("FRANCE_TRAVAIL_CLIENT_ID")
CLIENT_SECRET = os.getenv("FRANCE_TRAVAIL_CLIENT_SECRET")

print("Client ID chargé :", CLIENT_ID is not None)
print("Client Secret chargé :", CLIENT_SECRET is not None)

Client ID chargé : True
Client Secret chargé : True


In [121]:
print("Scope demandé :", data["scope"])
print("Scope obtenu :", token_data.get("scope"))

Scope demandé : api_stats-offres-demandes-emploiv1 offresetdemandesemploi
Scope obtenu : api_stats-offres-demandes-emploiv1 offresetdemandesemploi


In [122]:
# Construction de la requête OAuth afin de vérifier que notre authentification fonctionne

TOKEN_URL = "https://entreprise.francetravail.fr/connexion/oauth2/access_token?realm=/partenaire"

data = {
    "grant_type" :	"client_credentials" ,
    "client_id"	: CLIENT_ID,
    "client_secret"	: CLIENT_SECRET,
    "scope"	: "api_stats-offres-demandes-emploiv1 offresetdemandesemploi"

}

headers = {
    "Content-Type" : "application/x-www-form-urlencoded"
}

response = requests.post(TOKEN_URL, data = data , headers = headers)

if response_token.status_code == 200:
    token_data = response_token.json()
    TOKEN = token_data["access_token"]

    print("Token récupéré :", TOKEN is not None)
    print("Scope obtenu :", token_data.get("scope"))
    print("Expiration :", token_data.get("expires_in"), "secondes")
    
else:
    print("Erreur :", response_token.status_code)
    print(response_token.text)



Token récupéré : True
Scope obtenu : api_stats-offres-demandes-emploiv1
Expiration : 1499 secondes


In [123]:
# Vérifions que TOKEN contient maintenant le jeton d'accès

token_data = response.json()

TOKEN = token_data["access_token"]

print("Token obtenu :", TOKEN is not None)
print("Type :", token_data["token_type"])
print("Expiration :", token_data["expires_in"], "secondes")

print("Scope du token :", token_data["scope"])
print("Type du token :", token_data["token_type"])


Token obtenu : True
Type : Bearer
Expiration : 1499 secondes
Scope du token : api_stats-offres-demandes-emploiv1 offresetdemandesemploi
Type du token : Bearer


Nous allons commencer par récupérer les données des Endpoints de : 
- Catalogue Indicateur, 
- Offres et 
- Demandeurs.
Ainsi, Avant de collecter des statistiques, nous allons savoir quels indicateurs existent, comment ils sont définis et quels paramètres sont nécessaires pour les récupérer. 
Cela nous évitera de multiplier les appels API sans objectif précis.


In [124]:
# Nous allons appeler l'endpoint Catalogue indicateur de l'API Marché du travail 

response = requests.get(
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/referentiel/details-indicateurs",
    headers={
        "Authorization": f"Bearer {TOKEN}",
        "Accept": "application/json"
    },
    params={
        "codeFamille": "DEMANDEURS",
        "codeIndicateur": "DE_1"
    }
)

print("Status :", response.status_code)
print("URL appelée :", response.url)
print("Réponse :", response.text)

Status : 200
URL appelée : https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/referentiel/details-indicateurs?codeFamille=DEMANDEURS&codeIndicateur=DE_1
Réponse : {"listeIndicateurs":[{"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"ACC_1","libelleIndicateur":"Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)","descriptif":"Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .","codeFamille":"ACCESEMPLOI","disponibiliteIndicateur":1,"listeTypesValeurs":[{"datMaj":"2026-09-23T11:04:31.000+02:00","valeurPrincipale":"O","codeTypeValeur":"NB_ET_PCT","numOrdre":1,"libValeur":"Nombre et % de répartition de ce nombre par caractéristiques.","libelleValeurSpecifique":"Nombre et % des retours à l'emploi AB par durée d'emploi et caractéristiques"},{"

In [125]:

# Pour afficher le résultat de manière à ce qu'il soit lisible

import json

print(json.dumps(response.json(), indent=4, ensure_ascii=False))

{
    "listeIndicateurs": [
        {
            "datMaj": "2026-09-23T11:04:21.000+02:00",
            "codeIndicateur": "ACC_1",
            "libelleIndicateur": "Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)",
            "descriptif": "Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .",
            "codeFamille": "ACCESEMPLOI",
            "disponibiliteIndicateur": 1,
            "listeTypesValeurs": [
                {
                    "datMaj": "2026-09-23T11:04:31.000+02:00",
                    "valeurPrincipale": "O",
                    "codeTypeValeur": "NB_ET_PCT",
                    "numOrdre": 1,
                    "libValeur": "Nombre et % de répartition de ce nombre par caractéristiques.",
                    "libelleValeurSpecifique": "Nom

In [126]:
# Nous allons compter et expliquer les indicateurs que nous avons
## Convertissons la reponse json en objet python

catalogue = response.json()

## Vérifions la structure de la response

print("Clés disponibles: ", catalogue.keys())

## Comptons les indicateurs retournés

indicateurs = catalogue.get("listeIndicateurs", [])

print("Nombres indicateurs: ", len(indicateurs))

## Affichons les premiers indicateurs

for indicateur in indicateurs[:10] :
    print(
        indicateur.get("codeIndicateur"),
        indicateur.get("codeFamille"),
        indicateur.get("libelleIndicateur")
    )

Clés disponibles:  dict_keys(['listeIndicateurs'])
Nombres indicateurs:  19
ACC_1 ACCESEMPLOI Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)
ACC_2 ACCESEMPLOI Nombre et % des accès à l'emploi sorties formation par délai d'accès emploi x durée d'emploi x caractéristiques,Taux d'accès des formés  par caractéristiques
ACC_8 ACCESEMPLOI Nombre et % des retours à l'emploi AB des profils spécifiques x Catégorie Candidat , Taux de retour (= % par rapport aux AB)
CHOM_1 CHOMAGE Taux de chômage
DE_1 DEMANDEURS Nombre et % de DE par catégories x caractéristiques, % par activités (Rome, Compétence)
DE_3 DEMANDEURS Nombre et % de DE sorties formation par caractéristiques, % par activités (Rome, Formation, Compétence)
DE_5 DEMANDEURS Nombre de demandeurs d'emploi enregistrés en entrée (flux) + cumul des 12 derniers mois
DE_8 DEMANDEURS DE avec un profil spécifique par catégories (ABC / Toutes cat

In [127]:
# Nous allons afficher la liste des 19 indicateurs de l'API marché du travail

for indicateur in catalogue["listeIndicateurs"]:
    print("-"*80),
    print( "Code: ", indicateur.get("codeIndicateur")),
    print("Famille", indicateur.get("codeFamille")),
    print("Libéllé", indicateur.get("libelleIndicateur"))


--------------------------------------------------------------------------------
Code:  ACC_1
Famille ACCESEMPLOI
Libéllé Nombre et % des retours à l'emploi AB par durée d'emploi x caractéristiques, Taux de retour (= % par rapport aux AB) par caractéristiques (age, genre)
--------------------------------------------------------------------------------
Code:  ACC_2
Famille ACCESEMPLOI
Libéllé Nombre et % des accès à l'emploi sorties formation par délai d'accès emploi x durée d'emploi x caractéristiques,Taux d'accès des formés  par caractéristiques
--------------------------------------------------------------------------------
Code:  ACC_8
Famille ACCESEMPLOI
Libéllé Nombre et % des retours à l'emploi AB des profils spécifiques x Catégorie Candidat , Taux de retour (= % par rapport aux AB)
--------------------------------------------------------------------------------
Code:  CHOM_1
Famille CHOMAGE
Libéllé Taux de chômage
-----------------------------------------------------------------

Cette première version nous permet de recenser les données que nous choisirons pour répondre aux objectifs de notre observatoire de l'emploi. Les indicateurs que nous pourrons sélectionner pour un début sont ceux des offres, des demandeurs d'emploi et des embauches:

- `code:` CHOM_1 `codeFamille :` CHOMAGE et pour `libelleIndicateur: ` Taux de chômage `Utilité pour le projet: ` compléter l'analyse du marché du travail.

- `code:` DE_1 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre et % de DE par catégories x caractéristiques, % par activités (Rome, Compétence);`Utilité pour le projet: ` ce qui nous permettra de répartir les demandeurs d'emploi par catégories et par caractériques.

- `code:` DE_5 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre de demandeurs d'emploi enregistrés en entrée (flux) + cumul des 12 derniers mois;`Utilité pour le projet: ` ce qui nous permettra d'analyser le nombre des demandeurs d'emploi et cumul sur les 12 derniers mois.

-  `code:` DE_9 `codeFamille :` DEMANDEURS et pour `libelleIndicateur: ` Nombre et % de DE ayant retrouvé un emploi selon le metier recherché (FAP87,FAP21);`Utilité pour le projet: ` ce qui nous permettra de connaitre le nombre des demandeurs d'emploi ayant retrouvé un emploi selon le métier recherché.

- `code:` DYN_1 `codeFamille :` DYNEMPLOI et pour `libelleIndicateur: ` DYNEMPLOI Nous n'avons pas encore sa signification exacte, nous l'examinerons si plus amples informations.

- `code:` EMB_1 `codeFamille :` EMBAUCHES et pour `libelleIndicateur: `Nombre et % d'embauches par Catégorie Candidat x durée d'emploi x caractéristiques, % par activités (Rome, Secteur). `Utilité pour le projet: ` Etudier les embauches selon les caractéristiques et les activités.

- `code:` OFF_1 `codeFamille :` OFFRES et pour `libelleIndicateur: ` Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence). `Utilité pour le projet: ` nous allons étudier les offres par térritoire, métier et période.


In [128]:
# Nous allons chercher les informations associées à l'indicateur OFF_1 en affichant sa structure complète

for indicateur in catalogue["listeIndicateurs"]:
    if indicateur.get("codeIndicateur") == "OFF_1" : 
        print(json.dumps(indicateur, indent = 4, ensure_ascii=False))
        break

{
    "datMaj": "2026-09-23T11:04:21.000+02:00",
    "codeIndicateur": "OFF_1",
    "libelleIndicateur": "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)",
    "descriptif": "Définition détaillée : informations sur les sources de données utilisées, les méthodes de calcul, les limitations sur certains axes, les aspects réglementaires éventuels .",
    "codeFamille": "OFFRES",
    "disponibiliteIndicateur": 1,
    "listeTypesValeurs": [
        {
            "datMaj": "2026-09-23T11:04:31.000+02:00",
            "valeurPrincipale": "O",
            "codeTypeValeur": "NB_ET_PCT",
            "numOrdre": 1,
            "libValeur": "Nombre et % de répartition de ce nombre par caractéristiques.",
            "libelleValeurSpecifique": "Nombre et % d'offres par origine (Pole-emploi / Toutes offres) et caractéristiques"
        },
        {
            "datMaj": "2026-09-23T11:04:31.000+02:00",
       

Nous remarquons que OFF_1 contient énormément de paramètres et de combinaisons.

In [129]:
# Nous allons identifier les informations utiles: périodes, territoires, activités disponibles pour OFF_1

## retrouvons la définition complète de OFF_1
off_1 = next(
    item for item in catalogue["listeIndicateurs"]
    if item.get("codeIndicateur") == "OFF_1"
)
# print(off_1)

## Affichons les paramètres utiles pour préparer notre extraction
print("PERIODES")
print(json.dumps(off_1.get("listeTypesPeriodesIndicateur", []), indent =2, ensure_ascii=False))

print("TERRITOIRES")
print(json.dumps(off_1.get("listeTypesTerritoiresIndicateur", []), indent =2, ensure_ascii=False))

print("ACTIVITES")
print(json.dumps(off_1.get("listeTypesActivitesIndicateur", []), indent =2, ensure_ascii=False))

print("NOMMENCLATURES")
print(json.dumps(off_1.get("listeTypesNomenclaturesIndicateur", []), indent =2, ensure_ascii=False))

print("\nNOMBRE DE CROISEMENTS AUTORISÉS")
print(len(off_1.get("listeCroisementsIndicateur", [])))


PERIODES
[
  {
    "datMaj": "2026-09-23T11:04:30.000+02:00",
    "codeTypePeriode": "TRIMESTRE",
    "libelleTypePeriode": "Trimestre",
    "profondeurMax": "12"
  }
]
TERRITOIRES
[
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "NAT",
    "libelleTypeTerritoire": "Niveau National"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "REG",
    "libelleTypeTerritoire": "Région"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "DEP",
    "libelleTypeTerritoire": "Département"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "COM",
    "libelleTypeTerritoire": "Commune"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "BASBMO",
    "libelleTypeTerritoire": "Bassin (BMO)"
  },
  {
    "datMaj": "2026-09-23T11:04:23.000+02:00",
    "codeTypeTerritoire": "EPCI",
    "libelleTypeTerritoire": "Etablissement public de coopération interco

Nous apprenons ainsi que OFF_1 permet d'avoir:
- Périodes: Trimestres avec une profondeur maximale de 12.
- Territoires: National, Région, Département, Commune, bassin BMO, EPCI et comité local pour l'emploi.
- Activités: toutes activités, métier ROME, secteurs NAF, savoir-être, macro-compétence, filière ...
- Nomenclature: Origine des offres: France travail, partenaires ou toutes les offres.
- Croisement : 252 combinaisons autorisés. Ce sont des combinaisons possibles entre les caractéristiques. Nous allons sélectionner celles qui répondent à notre objectif métier.


Pour Construire un premier jeu de données exploitable,
nous allons commencer par une analyse des offres d'emploi par métier, par région et par trimestre.
Ce qui nous permettra de répondre à des questions concrètes telles que :
- Quels métiers présentent le plus d'offres d'emploi ?
- Comment le volume d'offre évolue t'il au fil des trimestres ?
- Quelles différences observe t-on entre les régions ?
- Les résultats changent-ils selon que l’on considère les offres France Travail ou toutes les offres ?

Nous avons déjà identifier dans les paramètres:
TRIMESTRE pour la période,
REG pour le térritoire région,
ROME pour le métier,
PE et TOFF pour distinguer l'origine des offres.

In [130]:
# Nous allons afficher les 5 premiers croisements et pas les 252

croisements = off_1.get("listeCroisementsIndicateur", [])

print("Nombre total de Croisements :", len(croisements))

for i, croisement in enumerate(croisements[:5], start = 1) : 
    print(f"\n --- Croisement {i} ---")
    print(json.dumps(croisement, indent = 2, ensure_ascii = False))


Nombre total de Croisements : 252

 --- Croisement 1 ---
{
  "datMaj": "2026-09-23T11:04:23.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE"
}

 --- Croisement 2 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE-CUMUL12MOIS"
}

 --- Croisement 3 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF"
}

 --- Croisement 4 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "BASBMO",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF-C

In [131]:
# Nous observons avec la précédente requête que le codeTypeTerritoire apparait par ordre analphabétique
# Pour vérifier cela, nous allons vérifier la répartition des 252 croisements
# Nous comptons le nombre de croisement par type de territoire

from collections import Counter

repartition = Counter(croisement.get("codeTypeTerritoire")
                        for croisement in croisements
                        )
print("Nombre de croisements par territoire: ")

for territoire, nombre in repartition.items():
    print(f"{territoire} : {nombre}")
    

Nombre de croisements par territoire: 
BASBMO : 36
CLPE : 36
COM : 36
DEP : 36
EPCI : 36
NAT : 36
REG : 36


In [132]:
# Observons dans quel ordre apparait les territoires dans les croisements sans doublons

territoires = [
                croisement.get("codeTypeTerritoire")
                for croisement in croisements
]

territoires_uniques = list(dict.fromkeys(territoires))

print("Ordre des territoires dans la liste :")
print(territoires_uniques)

Ordre des territoires dans la liste :
['BASBMO', 'CLPE', 'COM', 'DEP', 'EPCI', 'NAT', 'REG']


Nous observons de ce fait qu'il y'a 7 territoires et 36 croisements par territoire.

Les sept types de territoires sont :
BASBMO : Bassin BMO
CLPE: Comité local pour l'emploi
COM: Commune
DEP: Département
EPCI: Établissement public de coopération intercommunale
NAT: Niveau national
REG: Région

In [133]:
# Observons les 36 combinaisons d'un territoire - en fonction de RE - région
# Filtrer les croisements de type REG

croisements_reg = [
        croisement
        for croisement in croisements
        if croisement.get("codeTypeTerritoire") == "REG"
]

print("Nombre de croisements régionaux :", len(croisements_reg))

for i, croisement in enumerate(croisements_reg, start=1):
    print(f"\n--- Croisement {i} ---")
    print(json.dumps(croisement, indent=2, ensure_ascii=False))


Nombre de croisements régionaux : 36

--- Croisement 1 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE"
}

--- Croisement 2 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "PE-CUMUL12MOIS"
}

--- Croisement 3 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF"
}

--- Croisement 4 ---
{
  "datMaj": "2026-09-23T11:04:22.000+02:00",
  "codeTypeTerritoire": "REG",
  "codeTypeActivite": "CUMUL",
  "codeTypePeriode": "TRIMESTRE",
  "codeTypeNomenclature": "ORIGINEOFF",
  "codeNomenclature": "TOFF-CUMUL12MOIS"
}

Nous pouvons déduire des 36 croisements que :

Les croisements combinent 9 types d’activité × 4 nomenclatures, toujours avec les mêmes paramètres de période et de territoire. 
Les 9 types d’activité sont CUMUL, ETRE, FILIERE, MACRO_COMP, NAF21, NAF5, NAF88, OBJECTIF et ROME.

Les quatre valeurs de nomenclature sont :

PE : offres France Travail,
PE-CUMUL12MOIS : cumul sur les 12 derniers mois des offres France Travail,
TOFF : toutes les offres,
TOFF-CUMUL12MOIS : cumul sur les 12 derniers mois de toutes les offres.


In [134]:

import requests

url = (
    "https://api.francetravail.io/partenaire/stats-offres-demandes-emploi/v1/indicateur/stat-offres"
)

payload = {
    "codeTypeTerritoire": "REG",
    "codeTerritoire": "75",
    "codeTypeActivite": "ROME",
    "codeActivite": "A1203",
    "codeTypePeriode": "TRIMESTRE",
    "codeTypeNomenclature": "ORIGINEOFF",
    "codeNomenclature": "TOFF"
}

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

response = requests.post(
    url,
    json=payload,
    headers=headers
)

print("Statut :", response.status_code)
print("Type de contenu :", response.headers.get("Content-Type"))
print("Réponse :", response.text[:3000])

Statut : 200
Type de contenu : application/json
Réponse : {"datMaj":"2026-09-23T11:04:21.000+02:00","codeIndicateur":"OFF_1","codeFamille":"OFFRES","libIndicateur":"Nombre et % d'offres par origine (Pole-emploi / Toutes offres) x caractéristiques, % par activités (Rome, Secteur, Savoir-être, Compétence)","listeValeursParPeriode":[{"datMaj":"2026-09-23T09:06:21.000+02:00","codeTypeTerritoire":"REG","codeTerritoire":"75","libTerritoire":"NOUVELLE-AQUITAINE","codeTypeActivite":"ROME","codeActivite":"A1203","libActivite":"Agent / Agente d'entretien des espaces verts","codeNomenclature":"PE","libNomenclature":"Offres France Travail","codeTypePeriode":"TRIMESTRE","codePeriode":"2026T2","libPeriode":"2ème trimestre 2026","valeurPrincipaleNombre":290,"valeurSecondairePourcentage":0.0,"listeValeurParCaract":[{"codeTypeCaract":"TYPECTR","codeCaract":"CDI","libCaract":"CDI","nombre":60,"pourcentage":23.0,"masque":false},{"codeTypeCaract":"TYPECTR","codeCaract":"CDD1","libCaract":"CDD < 1 mois","n